# Hi-EF Phase 2: canonical hierarchical validation audit

This CPU-only notebook audits the frozen 4 × 5 canonical residual matrix. Attach the saved output of `hief-canonical-residual-matrix`, enable Internet, and choose **Save Version → Save & Run All**. It does not train models or load test data.

In [ ]:
from pathlib import Path
import json
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
OUTPUT = Path('/kaggle/working/canonical_residual_audit')

if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
else:
    subprocess.run([
        'git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'
    ], check=True)

candidates = []
for path in Path('/kaggle/input').rglob('canonical_residual_matrix_summary.json'):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        continue
    if payload.get('protocol') == 'canonical-contextual-affective-residual-validation-matrix-v1':
        candidates.append(path)
assert len(candidates) == 1, f'Expected exactly one canonical matrix output, found: {candidates}'
MATRIX = candidates[0].parent
MANIFEST = REPO / 'experiments/manifests/source_folder_split_seed42.csv'
commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Ready at commit:', commit)
print('Matrix input:', MATRIX)

In [ ]:
environment = {'PYTHONPATH': str(REPO / 'experiments')}
subprocess.run([
    'python', '-m', 'unittest',
    str(REPO / 'experiments/test_canonical_residual_audit.py'),
], cwd=REPO, env={**__import__('os').environ, **environment}, check=True)

subprocess.run([
    'python', str(REPO / 'experiments/analyze_canonical_residual_matrix.py'),
    '--matrix-dir', str(MATRIX),
    '--manifest', str(MANIFEST),
    '--output-dir', str(OUTPUT),
    '--bootstrap-replicates', '5000',
], check=True)

In [ ]:
import pandas as pd

summary_path = OUTPUT / 'canonical_residual_audit_summary.json'
effects_path = OUTPUT / 'canonical_audit_effects_by_seed.csv'
class_path = OUTPUT / 'canonical_audit_per_class_effects.csv'
summary = json.loads(summary_path.read_text())
assert summary['test_evaluated'] is False
assert summary['partitions_touched'] == ['validation']
display(pd.DataFrame(summary['hierarchical_bootstrap']).T)
print(json.dumps(summary['advancement_gate'], indent=2))
print(json.dumps(summary['claim_scope'], indent=2))
print('Download:', summary_path)
print('Download:', effects_path)
print('Download:', class_path)